In [ ]:
'''
* @name: metric.py
* @description: Evaluation metrics. Note: The code source from MMSA (https://github.com/thuiar/MMSA/tree/master).
'''

import numpy as np
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score

__all__ = ['AVEMetric']




class AVEMetric:
    def __init__(self, topk=None):
        self.topk = topk
        self.reset()

    def reset(self):
        """清空缓存（每个 epoch 调一次）"""
        self.y_pred = []
        self.y_true = []
        self.y_pred_probs = []  # 只在需要 top-k 时使用

    def update(self, y_pred, y_true, y_pred_probs=None):
        """
        流式更新
        :param y_pred: numpy array, (B,)
        :param y_true: numpy array, (B,)
        :param y_pred_probs: optional numpy array, (B, C)
        """
        self.y_pred.append(y_pred)
        self.y_true.append(y_true)

        if y_pred_probs is not None:
            self.y_pred_probs.append(y_pred_probs)

    def compute(self):
        """epoch 结束后统一计算指标"""
        y_pred = np.concatenate(self.y_pred, axis=0)
        y_true = np.concatenate(self.y_true, axis=0)

        results = {
            "accuracy": round(accuracy_score(y_true, y_pred), 4),
            "f1_score": round(f1_score(y_true, y_pred, average='weighted'), 4)
        }

        if len(self.y_pred_probs) > 0:
            y_pred_probs = np.concatenate(self.y_pred_probs, axis=0)
            for k in self.topk:
                try:
                    acc = top_k_accuracy_score(y_true, y_pred_probs, k=k)
                except Exception:
                    acc = 0.0
                results[f"top{k}_acc"] = round(acc, 4)

        return results

# prompt

In [ ]:
AVE_dict = {
    "Church bell": 0,
    "Male speech, man speaking": 1,
    "Bark": 2,
    "Fixed-wing aircraft, airplane": 3,
    "Race car, auto racing": 4,
    "Female speech, woman speaking": 5,
    "Helicopter": 6,
    "Violin, fiddle": 7,
    "Flute": 8,
    "Ukulele": 9,
    "Frying (food)": 10,
    "Truck": 11,
    "Shofar": 12,
    "Motorcycle": 13,
    "Acoustic guitar": 14,
    "Train horn": 15,
    "Clock": 16,
    "Banjo": 17,
    "Goat": 18,
    "Baby cry, infant cry": 19,
    "Bus": 20,
    "Chainsaw": 21,
    "Cat": 22,
    "Horse": 23,
    "Toilet flush": 24,
    "Rodents, rats, mice": 25,
    "Accordion": 26,
    "Mandolin": 27
}

KS_dict = {
    "dribbling basketball": 0,
    "tap dancing": 1,
    "playing harmonica": 2,
    "shoveling snow": 3,
    "singing": 4,
    "mowing lawn": 5,
    "tapping guitar": 6,
    "playing accordion": 7,
    "playing guitar": 8,
    "playing drums": 9,
    "playing trumpet": 10,
    "shuffling cards": 11,
    "playing bass guitar": 12,
    "playing trombone": 13,
    "playing bagpipes": 14,
    "blowing out candles": 15,
    "playing organ": 16,
    "playing saxophone": 17,
    "bowling": 18,
    "blowing nose": 19,
    "playing piano": 20,
    "playing violin": 21,
    "laughing": 22,
    "playing clarinet": 23,
    "tapping pen": 24,
    "chopping wood": 25,
    "playing xylophone": 26,
    "playing keyboard": 27,
    "ripping paper": 28,
    "tickling": 29,
    "stomping grapes": 30
}



UCF51_dict = {
    "ApplyEyeMakeup": 0,
    "ApplyLipstick": 1,
    "Archery": 2,
    "BabyCrawling": 3,
    "BalanceBeam": 4,
    "BandMarching": 5,
    "BasketballDunk": 6,
    "BlowDryHair": 7,
    "BlowingCandles": 8,
    "BodyWeightSquats": 9,
    "Bowling": 10,
    "BoxingPunchingBag": 11,
    "BoxingSpeedBag": 12,
    "BrushingTeeth": 13,
    "CliffDiving": 14,
    "CricketBowling": 15,
    "CricketShot": 16,
    "CuttingInKitchen": 17,
    "FieldHockeyPenalty": 18,
    "FloorGymnastics": 19,
    "FrisbeeCatch": 20,
    "FrontCrawl": 21,
    "Haircut": 22,
    "Hammering": 23,
    "HammerThrow": 24,
    "HandstandPushups": 25,
    "HandstandWalking": 26,
    "HeadMassage": 27,
    "IceDancing": 28,
    "Knitting": 29,
    "LongJump": 30,
    "MoppingFloor": 31,
    "ParallelBars": 32,
    "PlayingCello": 33,
    "PlayingDaf": 34,
    "PlayingDhol": 35,
    "PlayingFlute": 36,
    "PlayingSitar": 37,
    "Rafting": 38,
    "ShavingBeard": 39,
    "Shotput": 40,
    "SkyDiving": 41,
    "SoccerPenalty": 42,
    "StillRings": 43,
    "SumoWrestling": 44,
    "Surfing": 45,
    "TableTennisShot": 46,
    "Typing": 47,
    "UnevenBars": 48,
    "WallPushups": 49,
    "WritingOnBoard": 50
}

In [ ]:
import re

def extract_class_id_from_text(text, class_dict):

    # 提取所有整数（包括负号，虽然理论上不应该有）
    candidates = re.findall(r"-?\d+", text)

    if not candidates:
        return None

    valid_ids = set(class_dict.values())

    for c in candidates:
        cid = int(c)
        if cid in valid_ids:
            return cid

    return None


In [ ]:
def action_event_prompt_text_only(video_description, class_dict):

    class_list_str = "\n".join(
        [f"{v}: {k}" for k, v in sorted(class_dict.items(), key=lambda x: x[1])]
    )

    prompt = (
        "You are an expert action and event classification model.\n\n"
        "Your task is to identify the primary action or event described in the text below.\n\n"
        "You must rely ONLY on the provided textual description. "
        "Do NOT assume access to video frames, audio signals, or any additional context.\n\n"
        "Choose exactly ONE class from the following list and output its corresponding class ID:\n\n"
        f"{class_list_str}\n\n"
        "**Strict rules:**\n"
        "- Output ONLY one integer number\n"
        "- The number MUST be a valid class ID from the list above\n"
        "- Do NOT output the class name\n"
        "- Do NOT output any explanation, reasoning, or extra text\n\n"
        "Now classify the following video description:\n"
        f"\"{video_description}\""
    )

    return prompt




def action_event_prompt_video_only(class_dict):


    class_list_str = "\n".join(
        [f"{v}: {k}" for k, v in sorted(class_dict.items(), key=lambda x: x[1])]
    )

    prompt = (
        "You are an expert audio-visual action recognition model.\n\n"
        "Your task is to classify the primary action or event occurring in a video clip.\n\n"
        "You must rely ONLY on the following information:\n"
        "- Visual motion, objects, and human actions in the video\n"
        "- Audio cues such as sound events, speech, or environmental noise\n\n"
        "Choose exactly ONE class from the list below and output its corresponding class ID:\n\n"
        f"{class_list_str}\n\n"
        "**Strict rules:**\n"
        "- Output ONLY one integer number\n"
        "- The integer MUST be one of the class IDs listed above\n"
        "- Do NOT output the class name\n"
        "- Do NOT output any explanation, reasoning, or additional text\n\n"
        "Now output the class ID for this video:"
    )

    return prompt

Text_time = "In the video, the screen shows a woman with a neutral facial expression, slightly furrowed brows, and slightly downturned corners of the mouth, indicating that she may be feeling somewhat disengaged or unimpressed. Her gaze is direct, seemingly communicating with the other person. In the audio, the character's tone is low and monotonous, giving a sense of detachment. In the text, the subtitle says, ""And he was still boring."" This sentence is the woman's evaluation of someone, likely expressing her lack of interest or disappointment. Based on the woman's facial expression in the video, which suggests mild frustration or disengagement, as well as her direct gaze towards the other person, it can be inferred that she is conveying a sense of weariness or dissatisfaction. Additionally, the description of the character's low and monotonous tone in the audio also supports this inference. Therefore, this sentence carries a tone of subdued disappointment or apathy.","The woman conveys a sense of weariness or dissatisfaction through her neutral expression, slight frown, downturned mouth, and direct gaze, indicating mild frustration or disengagement."
action_event_prompt_video_only_AVE_text = action_event_prompt_text_only(Text_time, AVE_dict)
print(action_event_prompt_video_only_AVE_text)

print("\n\n====================\n\n")

action_event_prompt_video_only_AVE = action_event_prompt_video_only(KS_dict)
print(action_event_prompt_video_only_AVE)

# 调用qwen3-vl-32b-instruct (qwen-vl-max-latest)

In [ ]:
import re
import base64
import pandas as pd
from tqdm import tqdm
import pandas as pd
from openai import OpenAI
import os
tqdm.pandas()


client = OpenAI(
    # 若没有配置环境变量，请用百炼API Key将下行替换为：api_key="sk-xxx",
    api_key="sk-xxxxxx",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)


#  Base64 编码格式
def encode_video(video_path):
    with open(video_path, "rb") as video_file:
        return base64.b64encode(video_file.read()).decode("utf-8")


def QWen3_vl_32b_instruct_pridictor(video_path, question):

    # print(question)

    base64_video = encode_video(video_path)

    completion = client.chat.completions.create(
        model="qwen3-vl-32b-instruct",
        messages=[
            {
                "role": "system",
                "content": [{"type":"text","text": "You are a helpful assistant."}]},
            {
                "role": "user",
                "content": [
                    {
                        "type": "video_url",
                        "video_url": {"url": f"data:video/mp4;base64,{base64_video}"},
                    },
                    {"type": "text", "text": question},
                ],
            }
        ],
        extra_body={"enable_thinking": False},
    )
    
    # print(completion.choices[0].message.content)
    return completion.choices[0].message.content


In [ ]:
def QWen3_vl_32b_instruct_pridictor_with_subtitle(question):

    completion = client.chat.completions.create(
        # 模型列表：https://help.aliyun.com/zh/model-studio/getting-started/models
        # model="qwen-plus",
        model="qwen3-vl-32b-instruct",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"{question}"},
        ],
        # Qwen3模型通过enable_thinking参数控制思考过程（开源版默认True，商业版默认False）
        # 使用Qwen3开源版模型时，若未启用流式输出，请将下行取消注释，否则会报错
        # extra_body={"enable_thinking": False},
    )

    # print(completion.choices[0].message.content)
    return completion.choices[0].message.content


Text_time = "The video begins with a man riding a horse in an enclosed area, with the horse initially walking slowly and the man holding the reins. The horse then starts to trot, and the man continues to ride it around the enclosure. The horse's tail swishes back and forth as it moves. The wind blows, creating a distinct rustling sound, and the hooves of the horse clatter rhythmically on the ground amidst the gusts. A rhythmic pattern emerges, alternating between the galloping sound and the wind's constant presence. The clip-clop and wind combine into a soothing, steady rhythm as the horse continues to trot, with the man still holding the reins."
action_event_prompt_video_only_AVE_text = action_event_prompt_text_only(Text_time, AVE_dict)
# print(action_event_prompt_video_only_AVE_text)
response = QWen3_vl_32b_instruct_pridictor_with_subtitle(action_event_prompt_video_only_AVE_text)
print(response)

### 预测3个动作视听事件分类数据集

In [ ]:
def predict(data_csv_path, videos_path, dataSet_name, column_name='VideoID', data_dict=AVE_dict):

    output_csv_path = data_csv_path

    data = pd.read_csv(data_csv_path)

    if 'Qwen3-vl' not in data.columns:
        data['Qwen3-vl'] = None

    prossed_num = 0
    sample_pbar = tqdm(data.iterrows(), total=len(data), desc=f"Processing {dataSet_name} samples")
    for index, row in sample_pbar:
        video_id = row[column_name]
        video_file = f"{video_id}.mp4" if not video_id.endswith('.avi') else video_id
        video_path = os.path.join(videos_path, video_file)
        
        if not os.path.exists(video_path):
            print(f"Video file not found: {video_path}, skipping...")
            continue

        subtitle = row['Time_text']


        if row['Qwen3-vl'] is not None and not pd.isna(row['Qwen3-vl']):
            continue
        
        
        question = action_event_prompt_video_only(data_dict)
        # print(question)



        try:
            response = QWen3_vl_32b_instruct_pridictor(video_path, question)
        except Exception as e:
            try:
                question = action_event_prompt_text_only(subtitle, data_dict)
                response = QWen3_vl_32b_instruct_pridictor_with_subtitle(question)
            except Exception as e:
                response = "0" 
        
        # print("response: ", response)
        score = extract_class_id_from_text(response, data_dict)
        # print(f"Sample {index}: Extracted score: {score}, {row['Class_id']}")

        sample_pbar.set_postfix({"index": index, "score": score})

        data.at[index, 'Qwen3-vl'] = score
        prossed_num += 1

        if index % 100 == 0:
            data.to_csv(output_csv_path, index=False)
        
        # if prossed_num == 3:
        #     break
            

    data.to_csv(output_csv_path, index=False)
    print(f"Results saved to {output_csv_path}")

    return data

In [ ]:
ave_data = predict(
    data_csv_path='',
    videos_path='',
    dataSet_name='AVE',
    column_name='VideoID',
    data_dict=AVE_dict
)

In [ ]:
import pandas as pd
ave_data = pd.read_csv('')
true = ave_data['Class_id'].tolist()
pred = ave_data['Qwen3-vl'].tolist()
accuracy = accuracy_score(true, pred)
print(f"Accuracy: {accuracy}")

In [ ]:
ks_data = predict(
    data_csv_path='',
    videos_path='',
    dataSet_name='KS',
    column_name='youtube_id',
    data_dict=KS_dict
)

In [ ]:
import pandas as pd
ave_data = pd.read_csv('')
true = ave_data['Class_id'].tolist()
pred = ave_data['Qwen3-vl'].tolist()
accuracy = accuracy_score(true, pred)
print(f"Accuracy: {accuracy}")

In [ ]:
ucf51_data = predict(
    data_csv_path='',
    videos_path="",
    dataSet_name='UCF51',
    column_name='FilePath',
    data_dict=UCF51_dict
)

In [ ]:
import pandas as pd
ave_data = pd.read_csv('')
true = ave_data['Class_id'].tolist()
pred = ave_data['Qwen3-vl'].tolist()
accuracy = accuracy_score(true, pred)
print(f"Accuracy: {accuracy}")

# 调用GPPT-5.1 进行视频理解

In [ ]:
import cv2
import base64
from IPython.display import display, Image

def encode_frames(video_path, frames_num=3):
    base64Frames = []
    video = cv2.VideoCapture(video_path)

    if not video.isOpened():
        print(f"无法打开视频文件: {video_path}")
        return base64Frames

    # 获取视频总帧数
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

    # 计算均匀分布的帧索引
    frame_indices = [int(total_frames * i / (frames_num + 1)) for i in range(1, frames_num + 1)]

    # 定义目标尺寸
    target_width = 320
    target_height = 210

    current_frame = 0
    while video.isOpened():
        success, frame = video.read()
        if not success:
            break
        # 仅保存指定帧
        if current_frame in frame_indices:
            resized_frame = cv2.resize(frame, (target_width, target_height))
            _, buffer = cv2.imencode(".jpg", resized_frame)
            base64Frames.append(base64.b64encode(buffer).decode("utf-8"))
        current_frame += 1

    video.release()

    # print(len(base64Frames), "frames read.")
    # # 显示三帧图像
    # for i, frame in enumerate(base64Frames):
    #     print(f"Frame {i+1}:")
    #     display(Image(data=base64.b64decode(frame), format='jpg', width=320, height=320))

    return base64Frames


In [ ]:
import os
import re
import cv2
import base64
import time
from openai import OpenAI
from tqdm import tqdm
tqdm.pandas()

client = OpenAI(
    base_url='https://xiaoai.plus/v1',
    api_key='sk-xxxxxx',
)

def GPT_5_1_predictor(frames, prompt):

    PROMPT_MESSAGES = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
                *[{
                    "type": "image_url",
                    "image_url": {
                        "url": 'data:image/jpeg;base64,' + frame,
                    }
                } for frame in frames]
            ],
        },
    ]
    
    params = {
        "model": "gpt-5.1",
        "messages": PROMPT_MESSAGES,
        "max_tokens": 30, 
    }
    result = client.chat.completions.create(**params)
    return result.choices[0].message.content



In [ ]:
def predict_GPT5_1(data_csv_path, videos_path, dataSet_name, column_name='VideoID', data_dict=AVE_dict):

    output_csv_path = data_csv_path

    data = pd.read_csv(data_csv_path)

    if 'GPT-5_1' not in data.columns:
        data['GPT-5_1'] = None

    prossed_num = 0
    sample_pbar = tqdm(data.iterrows(), total=len(data), desc=f"Processing {dataSet_name} samples")
    for index, row in sample_pbar:


        if row['GPT-5_1'] is not None and not pd.isna(row['GPT-5_1']):
            continue


        video_id = row[column_name]
        video_file = f"{video_id}.mp4" if not video_id.endswith('.avi') else video_id
        video_path = os.path.join(videos_path, video_file)

        subtitle = row['Time_text']

        base64Frames = encode_frames(video_path, frames_num=3)

                
        question = action_event_prompt_video_only(data_dict)
        # print(question)

        
        try:
            response = GPT_5_1_predictor(base64Frames, question)
        except Exception as e:
            response = "0" 
        
        score = extract_class_id_from_text(response, data_dict)
        # print(f"Sample {index}: Extracted score: {score}, {row['Class_id']}")

        sample_pbar.set_postfix({"index": index, "score": score})

        data.at[index, 'GPT-5_1'] = score
        prossed_num += 1


        if index % 100 == 0:
            data.to_csv(output_csv_path, index=False)
        
        # if prossed_num == 3:
        #     break

    data.to_csv(output_csv_path, index=False)
    print(f"Results saved to {output_csv_path}")

    return data

In [ ]:
ave_data = predict_GPT5_1(
    data_csv_path='',
    videos_path='',
    dataSet_name='AVE',
    column_name='VideoID',
    data_dict=AVE_dict
)

In [ ]:
ave_data = pd.read_csv('')
true = ave_data['Class_id'].tolist()
pred = ave_data['GPT-5_1'].tolist()
accuracy = accuracy_score(true, pred)
print(f"Accuracy: {accuracy}")

In [ ]:
ks_data = predict_GPT5_1(
    data_csv_path='',
    videos_path='',
    dataSet_name='KS',
    column_name='youtube_id',
    data_dict=KS_dict
)

In [ ]:
import pandas as pd
ave_data = pd.read_csv('')
true = ave_data['Class_id'].tolist()
pred = ave_data['GPT-5_1'].tolist()
accuracy = accuracy_score(true, pred)
print(f"Accuracy: {accuracy}")

In [ ]:
ucf51_data = predict_GPT5_1(
    data_csv_path='',
    videos_path='',
    dataSet_name='UCF51',
    column_name='FilePath',
    data_dict=UCF51_dict
)

In [ ]:
import pandas as pd
ave_data = pd.read_csv('')
true = ave_data['Class_id'].tolist()
pred = ave_data['GPT-5_1'].tolist()
accuracy = accuracy_score(true, pred)
print(f"Accuracy: {accuracy}")